# Step1 · Extraction Pipeline

Download each selected company's 10-K filings and extract the **Item 1A (Risk Factors)**
section from each one.

Outputs:
- `data/item1a/{cik}_{year}.txt` — one cleaned Item 1A section per company-year
- `data/step1_filing_index.csv` — per-filing extraction status
- `data/step1_corpus.csv` — the document corpus (one row per filing) for preprocessing

In [1]:
# Auto-reload edited src/ modules without restarting the kernel
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from tqdm.auto import tqdm
from src import step1_edgar as edgar

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1 · Configuration

In [2]:
ROOT       = Path.cwd().parent          # project root (notebooks/ is one level down)
DATA_DIR   = ROOT / "data"
ITEM1A_DIR = DATA_DIR / "item1a"
ITEM1A_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 2010
END_YEAR   = 2025          # last fully-reported year; gives a complete 2010-2025 panel
FORCE      = True         # True = re-download & re-extract everything (use after
                           # changing edgar.extract_item_1a)
MIN_DOC_CHARS = 1500       # shorter extractions are treated as failures, not documents
print("Config ready.")

Config ready.


## 2 · Load the selected companies

In [ ]:
companies = pd.read_csv(DATA_DIR / "step1_selected_companies.csv", dtype={"cik": str})
companies["cik"] = companies["cik"].str.zfill(10)
print(f"{len(companies)} companies across {companies['sector'].nunique()} sectors")
companies.head()

110 companies across 11 sectors


,ticker,company,sector,cik,first_year,last_year,n_10k
0,CHTR,Charter Communications,Communication Services,0001091667,2010,2025,16
1,CMCSA,Comcast,Communication Services,0001166691,2010,2025,16
2,EA,Electronic Arts,Communication Services,0000712515,2010,2025,16
3,LYV,Live Nation Entertainment,Communication Services,0001335258,2010,2025,16
4,OMC,Omnicom Group,Communication Services,0000029989,2010,2025,16


## 3 · Run the pipeline

For each company: list its 10-Ks (full history) → download each → extract Item 1A → save.
Already-extracted files are skipped unless `FORCE=True`.

In [ ]:
def run_pipeline(companies, start_year, end_year, force=False):
    records = []
    for _, company in tqdm(companies.iterrows(), total=len(companies), desc="Companies"):
        cik, ticker, name = company["cik"], company["ticker"], company["company"]
        try:
            filings = edgar.list_10k_filings(cik, start_year, end_year)
        except Exception as e:
            print(f"  {ticker}: could not list filings ({e})")
            continue

        for _, filing in filings.iterrows():
            year = int(filing["year"])
            out_path = ITEM1A_DIR / f"{cik}_{year}.txt"
            base = {"ticker": ticker, "company": name, "cik": cik, "year": year,
                    "accession": filing["accession_number"]}

            if out_path.exists() and not force:
                records.append({**base, "status": "skipped", "path": str(out_path.relative_to(ROOT))})
                continue
            try:
                html = edgar.download_filing(cik, filing["accession_number"], filing["primary_document"])
            except Exception as e:
                records.append({**base, "status": f"download_failed: {e}", "path": None})
                continue

            section = edgar.extract_item_1a(html)
            if section and len(section) >= MIN_DOC_CHARS:
                out_path.write_text(section, encoding="utf-8")
                records.append({**base, "status": "success", "path": str(out_path.relative_to(ROOT))})
            else:
                records.append({**base, "status": "item1a_not_found", "path": None})
    return pd.DataFrame(records)


index = run_pipeline(companies, START_YEAR, END_YEAR, force=FORCE)
index.to_csv(DATA_DIR / "step1_filing_index.csv", index=False)
print("\nStatus counts:")
print(index["status"].value_counts().to_string())

Companies: 100%|██████████| 110/110 [32:58<00:00, 17.99s/it]


Status counts:
status
success             1661
item1a_not_found      99


## 4 · Extraction summary

In [5]:
ok = index[index["status"] == "success"]
print(f"Successful extractions : {len(ok)} / {len(index)} filings")
print(f"Companies              : {ok['cik'].nunique()}")
print(f"Year range             : {ok['year'].min()}-{ok['year'].max()}")
print("\nSuccessful extractions per year:")
print(ok.groupby("year").size().sort_index().to_string())

Successful extractions : 1661 / 1760 filings
Companies              : 109
Year range             : 2010-2025

Successful extractions per year:
year
2010     97
2011    104
2012    103
2013    103
2014    105
2015    106
2016    103
2017    105
2018    105
2019    104
2020    104
2021    104
2022    104
2023    104
2024    105
2025    105


## 5 · Build the corpus

Collect successful extractions into `step1_corpus.csv` (one row per filing). The length guard
in the pipeline already excluded truncated sections, so every row here is a real Item 1A.

In [ ]:
def load_corpus(index):
    rows = []
    for _, r in index[index["path"].notna()].iterrows():
        text = (ROOT / r["path"]).read_text(encoding="utf-8").strip()
        rows.append({"cik": r["cik"], "ticker": r["ticker"], "company": r["company"],
                     "year": r["year"], "text": text})
    return pd.DataFrame(rows)


corpus = load_corpus(index)
corpus.to_csv(DATA_DIR / "step1_corpus.csv", index=False)
print(f"step1_corpus.csv: {corpus.shape[0]} filings, {corpus['cik'].nunique()} companies, "
      f"{corpus['year'].min()}-{corpus['year'].max()}")
corpus.head()

step1_corpus.csv: 1661 filings, 109 companies, 2010-2025


,cik,ticker,company,year,text
0,0001091667,CHTR,Charter Communications,2010,Risks Related to Our Emergence From\nBankruptc...
1,0001091667,CHTR,Charter Communications,2011,Risks Related to Our Significant Indebtedness\...
2,0001091667,CHTR,Charter Communications,2012,Risks Related to Our Indebtedness \nWe have a ...
3,0001091667,CHTR,Charter Communications,2013,Risks Related to Our Indebtedness \nWe have a ...
4,0001091667,CHTR,Charter Communications,2014,Risks Related to Our Indebtedness \nWe have a ...


---
Next: run `python -m src.step2_preprocess` and `python -m src.step2_keyword_filter` to clean, chunk, and filter `step1_corpus.csv` (QA report: `step2_preprocessing_qa.ipynb`).